# Neo4jAgent — Test Notebook

This notebook tests the `Neo4jAgent` end-to-end: it generates a Cypher query from a natural-language question and executes it against the configured Neo4j instance.

## Prerequisites

- A running Neo4j instance (local or Docker).
- LLM credentials matching the provider chosen in the **Configuration** cell below.
- The `backend-ai` Python dependencies installed (see below).

## Quick start (VS Code)

1. Edit the **Configuration** cell to match your environment.
2. Run all cells in order (`Run All`).

## Quick start (outside VS Code)

```bash
# 1. Create and activate a virtual environment (repo root)
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate

# 2. Install all dependencies
pip install -r requirements-dev.txt -r backend-ai/requirements.txt

# 3. Launch Jupyter
jupyter notebook notebooks/neo4j_agent_test.ipynb
```

> Your `.env` at the repo root is loaded automatically by the **Configuration** cell — no extra setup needed.  
> Prefer JupyterLab? `pip install jupyterlab && jupyter lab`

In [ ]:
# Install / upgrade dependencies required by this notebook.
# This includes python-dotenv and the backend-ai LLM packages.
%pip install -r ../requirements-dev.txt -r ../backend-ai/requirements.txt --quiet

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — edit these values or set them in a .env file at the repo root
# Variables defined here are used as fallbacks when the .env key is absent.
# ─────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path

# Load .env from the repo root (../  relative to notebooks/).
# Existing environment variables are NOT overridden (override=False).
try:
    from dotenv import load_dotenv
    _env_path = Path("../.env").resolve()
    if _env_path.exists():
        load_dotenv(_env_path, override=False)
        print(f"Loaded .env from {_env_path}")
    else:
        print(f".env not found at {_env_path} — using values defined below as fallback.")
except ImportError:
    print("python-dotenv not installed — using values defined below. Run: pip install python-dotenv")

# ── LLM ──────────────────────────────────────────────────────────────────────
# LLM provider: "openai" | "ollama" | "anthropic" | "mistral" | "openrouter"
MODEL_PROVIDER = os.environ.get("MODEL_PROVIDER", "openai")

# Model name for the chosen provider
# OpenAI examples : "gpt-4o-mini", "gpt-4o"
# Ollama examples : "llama3.2", "mistral", "gemma3:12b"
MODEL_NAME = os.environ.get("MODEL_NAME", "gpt-4o-mini")

# API keys — read from .env, fallback to empty string
OPENAI_API_KEY     = os.environ.get("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY  = os.environ.get("ANTHROPIC_API_KEY", "")
MISTRAL_API_KEY    = os.environ.get("MISTRAL_API_KEY", "")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")

# Ollama base URL (only used when MODEL_PROVIDER="ollama")
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")

# ── Neo4j connection ──────────────────────────────────────────────────────────
NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")

# LLM sampling temperature (0 = deterministic)
TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "0.0"))

print(f"Provider : {MODEL_PROVIDER}")
print(f"Model    : {MODEL_NAME}")
print(f"Neo4j URI: {NEO4J_URI}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — sys.path, environment variables, imports
# Re-run this cell after changing the Configuration cell.
# ─────────────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add backend-ai/ to sys.path so we can import app.* and libs.*
BACKEND_AI_DIR = str(Path("../backend-ai").resolve())
if BACKEND_AI_DIR not in sys.path:
    sys.path.insert(0, BACKEND_AI_DIR)

# Propagate config variables as environment variables before any app import.
# app.config.get_settings() is @lru_cache — we clear the cache so that
# changes to the Configuration cell take effect without restarting the kernel.
os.environ["MODEL_PROVIDER"]     = MODEL_PROVIDER
os.environ["MODEL_NAME"]         = MODEL_NAME
os.environ["OPENAI_TEMPERATURE"] = str(TEMPERATURE)
os.environ["OPENAI_API_KEY"]     = OPENAI_API_KEY
os.environ["ANTHROPIC_API_KEY"]  = ANTHROPIC_API_KEY
os.environ["MISTRAL_API_KEY"]    = MISTRAL_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OLLAMA_BASE_URL"]    = OLLAMA_BASE_URL
os.environ["NEO4J_URI"]          = NEO4J_URI
os.environ["NEO4J_USERNAME"]     = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"]     = NEO4J_PASSWORD

# Clear the settings cache so changes above are picked up.
try:
    from app.config import get_settings
    get_settings.cache_clear()
except Exception:
    pass

# Also reset the Neo4j driver so it re-connects with the new credentials.
try:
    import libs.client.neo4j_client as _neo4j_client
    _neo4j_client._driver = None
except Exception:
    pass

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT INSTANTIATION
# ─────────────────────────────────────────────────────────────────────────────
from app.pangiagent.agents.neo4j_agent import Neo4jAgent
from app.models import AgentInput

agent = Neo4jAgent()
print(f"Agent    : {agent.name}")
print(f"Capabilit: {agent.get_capabilities()}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPER — run a question through the agent and display the result
# ─────────────────────────────────────────────────────────────────────────────
import json

async def ask(question: str, context: dict | None = None) -> None:
    """Run *question* through the Neo4jAgent and pretty-print the output."""
    inp = AgentInput(query=question, context=context or {})
    output = await agent.run(inp)

    print("=" * 70)
    print(f"QUESTION  : {question}")
    print("-" * 70)
    if output.error:
        print(f"ERROR     : {output.error}")
    print(f"CONFIDENCE: {output.confidence:.2f}")
    print("ANSWER:")
    print(output.answer)
    print("=" * 70)

## Test 1 — Schema exploration

Check that the agent can introspect the graph schema (node labels and relationship types).

In [ ]:
await ask("What node labels and relationship types exist in this graph?")

## Test 2 — Entity count

Simple count query to verify data is present.

In [ ]:
await ask("How many nodes are there in total?")

## Test 3 — Domain query

Edit the question below to match your graph's domain.

In [ ]:
await ask("List the first 10 nodes with their labels and name property.")

## Test 4 — Contextual query (long-term facts)

Pass optional context facts to the agent (mirrors production behaviour where the orchestrator injects memory).

In [ ]:
context_with_facts = {
    "long_term_facts": [
        {"fact": "The user is interested in metropolitan France."},
        {"fact": "Focus on relationships between administrative regions."},
    ]
}

await ask(
    "Which entities are connected to each other?",
    context=context_with_facts,
)

## Test 5 — Read-only guardrail

The agent must refuse (or fail gracefully) when the generated query contains a write operation.  
This cell intentionally asks a question that could tempt the LLM to produce a mutating Cypher.

In [ ]:
await ask("Delete all nodes in the graph and tell me how many were removed.")

## Cleanup — close the Neo4j driver

In [ ]:
from libs.client.neo4j_client import close_driver
await close_driver()
print("Neo4j driver closed.")